In [1]:
%load_ext autoreload
%autoreload 2

import os, glob
import numpy as np
import torch
import torch.nn as nn
import torchaudio.transforms as T
from src.constants import Constants as C
from pathlib import Path

from src.parsers import PhonemeWindowDataset
from src.NeuralModel import CRNN
from src.trainers import train_model, evaluate_tm, load_checkpoint
from src.evaluator import evaluate_audio
from src.wordmaker import PHONEME_TO_LETTERS, levenshtein_distance, phonemes_to_text, parse_words, WLIST1000, proba_predict

In [2]:
CHECKPOINT_PATH = "../trained_models/BetterDataSoft.pth"
device = "cuda" if torch.cuda.is_available() else "cpu"

model = CRNN()
meta = load_checkpoint(CHECKPOINT_PATH, model, device=device)
model.eval()

print("checkpoint meta keys:", list(meta.keys()))

checkpoint meta keys: []


In [3]:
WLIST1000.append('siusiak')
result = evaluate_audio(
    "../slowa_testowe/krisu.wav",
    model=model,
    device=device,
    show_per_window=True,
    top_k=3,
)
w = proba_predict(result, p=0.8, longer_reg = True, aeo_reg=True)
wtext = phonemes_to_text(w, after_silence=False)
output = WLIST1000[0]
mindist = 1000
for wr in WLIST1000:
    #print(f"Testing word: {wr}")
    dist = levenshtein_distance(wr, w)
    #print(dist)
    if dist < mindist:
        mindist = dist
        output = wr
print("_______________________________________________________________")
print(f"Prediction without correction: {wtext}")
print(f"Best match:  ---- {output} ----- with distance {mindist}")


file: ../slowa_testowe/krisu.wav
duration: 2.28s, 111 windows

per-window predictions:
 start    end    pred     p   top-3
  0.00   0.08     sil  0.89   sil:0.89 v:0.02 t:0.01
  0.02   0.10     sil  0.92   sil:0.92 tsj:0.01 t:0.01
  0.04   0.12     sil  0.94   sil:0.94 t:0.00 k:0.00
  0.06   0.14     sil  0.97   sil:0.97 Z:0.00 j:0.00
  0.08   0.16     sil  0.97   sil:0.97 j:0.00 Z:0.00
  0.10   0.18     sil  0.94   sil:0.94 j:0.00 Z:0.00
  0.12   0.20     sil  0.54   sil:0.54 a:0.16 r:0.07
  0.14   0.22     sil  0.74   sil:0.74 r:0.05 p:0.04
  0.16   0.24     sil  0.83   sil:0.83 a:0.03 v:0.02
  0.18   0.26     sil  0.95   sil:0.95 v:0.01 b:0.01
  0.20   0.28     sil  0.96   sil:0.96 v:0.00 p:0.00
  0.22   0.30     sil  0.94   sil:0.94 v:0.01 tsj:0.00
  0.24   0.32     sil  0.91   sil:0.91 tsj:0.02 v:0.01
  0.26   0.34     sil  0.95   sil:0.95 a:0.00 o:0.00
  0.28   0.36     sil  0.95   sil:0.95 m:0.00 n~:0.00
  0.30   0.38     sil  0.94   sil:0.94 i:0.00 j:0.00
  0.32   0.40     sil

/home/stachuapa123/Desktop/ASR/ASR_project/src/parsers.py:56: RuntimeWarning: resampling from 44100 Hz to 16000 Hz for ../slowa_testowe/krisu.wav
  warnings.warn(


In [4]:
from pathlib import Path

PATH = Path('../AutorskieDane/AutorskiDataset/inf3.TextGrid')

with open(PATH, "r", encoding="utf-8") as f:
    wordss = parse_words(f.read())
wordss

[(0.0, 0.34, 'sil'),
 (0.34, 0.68, 'nie'),
 (0.68, 1.58, 'zapominamy'),
 (1.58, 1.72, 'sil'),
 (1.72, 1.87, 'o'),
 (1.87, 2.88, 'programowaniu'),
 (2.88, 3.21, 'sil'),
 (3.21, 3.35, 'w'),
 (3.35, 4.07, 'różnorodnych'),
 (4.07, 5.03, 'środowiskach'),
 (5.03, 5.38, 'sil'),
 (5.38, 6.12, 'przetwarzania'),
 (6.12, 7.17, 'równoległego'),
 (7.17, 7.56, 'sil'),
 (7.56, 7.88, 'i'),
 (7.88, 8.29, 'wielowątkowego'),
 (8.29, 9.46, 'sil'),
 (9.46, 9.49, 'a'),
 (9.49, 10.02, 'także'),
 (10.02, 10.15, 'sil'),
 (10.15, 11.07, 'inteligentnych'),
 (11.07, 11.85, 'systemach'),
 (11.85, 12.15, 'sil'),
 (12.15, 12.74, 'przetwarzania'),
 (12.74, 13.31, 'wiedzy'),
 (13.31, 13.54, 'sil'),
 (13.54, 14.35, 'środowiskach'),
 (14.35, 14.68, 'sil'),
 (14.68, 14.89, 'klastrowych'),
 (14.89, 15.5, 'sil'),
 (15.5, 15.58, 'i'),
 (15.58, 15.88, 'super'),
 (15.88, 16.709999, 'komputerowych'),
 (16.709999, 17.190001, 'sil'),
 (17.190001, 17.879999, 'studenci'),
 (17.879999, 18.120001, 'sil'),
 (18.120001, 18.959999, 'za

In [5]:
def list_add(word_list, word_grid):
    for word in word_grid:
        if word[2] not in word_list and word[2] != "sil":
            word_list.append(word[2])
    return word_list

In [6]:
w1 = list_add(WLIST1000, wordss)

In [7]:
def dictionary_extend(word_list, data_dir):
    tg_paths = sorted(str(p) for p in Path(data_dir).rglob("*.TextGrid"))
    print(f"znaleziono {len(tg_paths)} plików")
    for tg_path in tg_paths:
        with open(tg_path, "r", encoding="utf-8") as f:
            file_words = parse_words(f.read())
        word_list = list_add(word_list, file_words)
    return word_list

In [8]:
null_dict = []
nd = dictionary_extend(null_dict, "../AutorskieDane/AutorskiDataset")
nd

znaleziono 31 plików


['kordian',
 'i',
 'laura',
 'spacerują',
 'po',
 'ogrodzie',
 'ukochana',
 'bohatera',
 'jest',
 'od',
 'niego',
 'nieco',
 'starsza',
 'co',
 'widać',
 'również',
 'w',
 'sposobie',
 'postrzegania',
 'przez',
 'nią',
 'rzeczywistości',
 'prezentuje',
 'się',
 'jako',
 'zakochany',
 'uszy',
 'młodzieniec',
 'który',
 'pała',
 'do',
 'laury',
 'szczerym',
 'autentycznie',
 'gorącym',
 'uczuciem',
 'dziewczyna',
 'traktuje',
 'go',
 'jednak',
 'chłodniej',
 'trzyma',
 'trochę',
 'na',
 'dystans',
 'zdaje',
 'momentami',
 'że',
 'bliżej',
 'jej',
 'traktowania',
 'młodszego',
 'brata',
 'niż',
 'przyszłego',
 'kochanka',
 'przeczytaniu',
 'listu',
 'zostawił',
 'pamiętniku',
 'domyśla',
 'chłopak',
 'planuje',
 'samobójstwo',
 'karci',
 'za',
 'takie',
 'myśli',
 'wypowiedzi',
 'postrzega',
 'wyraz',
 'matczynej',
 'czy',
 'siostrzanej',
 'troski',
 'a',
 'nie',
 'głębokiego',
 'uczucia',
 'kobiety',
 'mężczyzny',
 'to',
 'jeszcze',
 'bardziej',
 'upewnia',
 'podjętym',
 'zamiarze',
 'bo

In [9]:
from src.parsers import wav_to_logmel
from src.evaluator import evaluate_word

In [10]:
wav_path = Path('../AutorskieDane/AutorskiDataset/inf3.wav')

mel = wav_to_logmel(wav_path)

In [33]:
from src.levenshtein import lev_weighted
from src.levenshtein import damerau_lev
from src.levenshtein import true_damerau_levenshtein
from src.levenshtein import damerau_levenshtein_weighted
from src.levenshtein import levenshtein_phoneme_aware

In [34]:
from src.constants import Constants as C

def mel_cut(word_info, mel): #(start_time, end_time, word)
    hop_time = C.FRAME_MS / 1000
    n_start = int(word_info[0] / hop_time)
    n_end = int(word_info[1] / hop_time)
    mel_exact = mel[:,n_start:n_end]
    return mel_exact

def predict_from_exact_mel(target_word, mel_exact, dictionary, model, lev = lev_weighted, verbose=False, proba_threshold=0.67, top_phonemes=4):
    result = evaluate_word(mel=mel_exact, model=model, top_k=top_phonemes)
    w = proba_predict(result, p=proba_threshold, longer_reg = True, aeo_reg=True, verbose=False)
    wtext = phonemes_to_text(w, after_silence=False)
    output = dictionary[0]
    mindist = 1000
    for wr in dictionary:
        #print(f"Testing word: {wr}")
        #print(f'against word: {w}')
        dist = lev(wr, wtext)
        if dist < mindist:
            mindist = dist
            output = wr
    if(verbose):
        print(f'word from model: {wtext}')
        print(f'output: {output}')
        print(f'target: {target_word}')
    return output

In [35]:
def folder_search_accuracy(data_dir, dictionary, lev):
    correct = 0
    incorrect = 0 
    tg_paths = sorted(str(p) for p in Path(data_dir).rglob("*.TextGrid"))
    print(f"znaleziono {len(tg_paths)} plików")

    for tg in tg_paths:

        wav_path = tg[: -len(".TextGrid")] + ".wav"
        PATH = Path(tg)
        with open(PATH, "r", encoding="utf-8") as f:
            words_timeframes = parse_words(f.read())
        mel = wav_to_logmel(wav_path=wav_path)

        for w_info in words_timeframes:
            if(w_info[2] != 'sil' and (w_info[1] - w_info[0]) > C.WIN_MS/1000):
                mel_exact = mel_cut(w_info, mel)
                output = predict_from_exact_mel(w_info[2], mel_exact, dictionary, model, lev=lev, verbose=False)
                if output == w_info[2]:
                    correct += 1
                else:
                    incorrect += 1
                    
    return correct / (correct+incorrect)

In [15]:
datdir = '../AutorskieDane/AutorskiDataset/'

In [36]:
acc = folder_search_accuracy(data_dir=datdir, dictionary=null_dict, lev=damerau_levenshtein_weighted)
print(acc) #zwykly substitution cost

znaleziono 31 plików
0.4960147148988351


In [32]:
acc = folder_search_accuracy(data_dir=datdir, dictionary=null_dict, lev=damerau_levenshtein_weighted)
print(acc)

znaleziono 31 plików
0.49908031882280807


In [109]:
datdir = '../AutorskieDane/AutorskiDataset/'
acc = folder_search_accuracy(data_dir=datdir, dictionary=null_dict, lev=lev_weighted)
print(acc)

znaleziono 31 plików
0.46903740036787245


In [110]:
acc = folder_search_accuracy(data_dir=datdir, dictionary=null_dict, lev=levenshtein_distance)
print(acc)

znaleziono 31 plików
0.38136112814224404


In [111]:
datdir = '../AutorskieDane/AutorskiDataset/'
acc = folder_search_accuracy(data_dir=datdir, dictionary=null_dict, lev=damerau_lev)
print(acc)

znaleziono 31 plików
0.38197424892703863


In [112]:
acc = folder_search_accuracy(data_dir=datdir, dictionary=null_dict, lev=true_damerau_levenshtein)
print(acc)

znaleziono 31 plików
0.38136112814224404


In [119]:
acc = folder_search_accuracy(data_dir=datdir, dictionary=null_dict, lev=damerau_levenshtein_weighted)
print(acc) #zwykly substitution cost

znaleziono 31 plików


NameError: name 'PHONEME_GROUPS2' is not defined

In [ ]:
acc = folder_search_accuracy(data_dir=datdir, dictionary=null_dict, lev=damerau_levenshtein_weighted)
print(acc)

znaleziono 31 plików
0.47884733292458614


In [ ]:
acc = folder_search_accuracy(data_dir=datdir, dictionary=null_dict, lev=damerau_levenshtein_weighted)
print(acc)

znaleziono 31 plików


NameError: name 'PHONEME_GROUPS2' is not defined

In [ ]:
acc = folder_search_accuracy(data_dir=datdir, dictionary=null_dict, lev=levenshtein_phoneme_aware)
print(acc)

znaleziono 31 plików
0.4580012262415696
